# P2-04 · Agente ReAct + Reflecting con LangGraph
**Trabajo Práctico 2 — Ingeniería Ontológica 3010090**

Este notebook implementa el núcleo del sistema RAG agéntico:

## Arquitectura de agentes

### ReAct (Reasoning + Acting)
El agente **razona** sobre qué herramientas usar y **actúa** invocándolas. A diferencia del flujo determinista de P1, aquí el agente **decide autónomamente** el plan de acción.

### Reflecting (Agente Reflexivo)
Después de generar una respuesta, el sistema la **evalúa/critica**. Si no cumple los criterios de calidad, **regenera** la respuesta (máximo 3 intentos). Tras 3 fallos, hace búsqueda en internet y retroalimenta la base de conocimiento.

## Tools disponibles para el agente
| Tool | Función |
|---|---|
| `vector_search` | Búsqueda MMR en FAISS |
| `knowledge_graph_query` | Consultas SPARQL a GraphDB |
| `apply_hyde` | Transformación HyDE para consultas vagas |
| `decompose_query` | Descomposición de consultas complejas |
| `internet_search` | Búsqueda web (Tavily) como fallback |

In [ ]:
# ── Configuración local (reemplaza google.colab) ──────────────────────────
import sys
from pathlib import Path
# Ruta al repo resuelta desde este notebook (funciona para cualquier colaborador)
_REPO_ROOT = Path('__file__').parent.parent if '__file__' in dir() else Path().resolve()
# Buscar local_config.py subiendo directorios si hace falta
for _p in [_REPO_ROOT, _REPO_ROOT.parent, Path().resolve(), Path().resolve().parent]:
    if (_p / 'local_config.py').exists():
        _REPO_ROOT = _p
        break
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))
from local_config import (
    BASE_DIR, CORPUS_DIR, INDEX_DIR, TTL_PATH,
    GRAPHDB_BASE, REPO_NAME, GRAPHDB_URL,
    GOOGLE_API_KEY, GROQ_API_KEY, LANGCHAIN_API_KEY, TAVILY_API_KEY
)
print('✅ Configuración local cargada')
print(f'   BASE_DIR:    {BASE_DIR}')
print(f'   CORPUS_DIR:  {CORPUS_DIR}')
print(f'   GRAPHDB_URL: {GRAPHDB_URL}')


## 1 · Definición de Tools (herramientas del agente)

In [ ]:
import os
from pathlib import Path
from typing import List, Dict, Any, Annotated, TypedDict, Optional
from datetime import datetime
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from SPARQLWrapper import SPARQLWrapper, JSON as SPARQL_JSON
faiss_index = FAISS.load_local(
    allow_dangerous_deserialization=True
)
groq   = ChatGroq(model='llama-3.3-70b-versatile', temperature=0.0, max_tokens=512)

# ── Modelos ─────────────────────────────────────────────────────────────
embeddings = GoogleGenerativeAIEmbeddings(model='models/text-embedding-004', task_type='retrieval_document')
gemini = ChatGoogleGenerativeAI(model='gemini-2.0-flash', temperature=0.2)
groq   = ChatGroq(model='llama-3.3-70b-versatile', temperature=0.0, max_tokens=512)

print('✅ Imports y modelos listos')


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# TOOL 1: Búsqueda vectorial MMR en FAISS
# ────────────────────────────────────────────────────────────────────────────
@tool
def vector_search(query: str, k: int = 5) -> str:
    """
    Realiza una búsqueda semántica con MMR en el corpus biomédico indexado (FAISS).
    Devuelve los fragmentos más relevantes y diversos del corpus.
    Usar cuando la pregunta requiere información factual de los artículos científicos.
    """
    retriever = faiss_index.as_retriever(
        search_type='mmr',
        search_kwargs={'k': k, 'fetch_k': k * 4, 'lambda_mult': 0.6}
    )
    docs = retriever.invoke(query)
    
    results = []
    for i, doc in enumerate(docs):
        results.append(
            f'[Fragmento {i+1}]\n'
            f'Fuente: {doc.metadata["doc_id"]}\n'
            f'Chunk: {doc.metadata["chunk_id"]}\n'
            f'Contenido: {doc.page_content[:600]}'
        )
    
    return '\n\n'.join(results) if results else 'No se encontraron documentos relevantes.'


# ────────────────────────────────────────────────────────────────────────────
# TOOL 2: Consulta al Knowledge Graph (GraphDB via SPARQL)
# ────────────────────────────────────────────────────────────────────────────
@tool
def knowledge_graph_query(entity: str, relation_type: str = 'all') -> str:
    """
    Consulta la ontología biomédica en GraphDB mediante SPARQL.
    Extrae relaciones estructuradas sobre genes, proteínas, enfermedades y fármacos.
    Usar para preguntas sobre relaciones semánticas entre entidades biomédicas.
    
    Args:
        entity: nombre de la entidad a consultar (ej. 'BRCA1', 'BreastCancer', 'Trastuzumab')
        relation_type: tipo de relación ('treats', 'encodes', 'studiesDisease', 'all')
    """
    GRAPHDB_URL = 'http://localhost:7200/repositories/biomed-kg'
    PREFIX = 'http://www.unal.edu.co/biomed#'
    
    try:
        sparql = SPARQLWrapper(GRAPHDB_URL)
        sparql.setReturnFormat(SPARQL_JSON)
        
        if relation_type == 'treats':
            query = f"""
            PREFIX bio: <{PREFIX}>
            SELECT ?subject ?predicate ?object WHERE {{
                ?subject ?predicate ?object .
                FILTER(CONTAINS(STR(?subject), '{entity}') || CONTAINS(STR(?object), '{entity}'))
                FILTER(?predicate = bio:treats || ?predicate = bio:isTreatedBy)
            }} LIMIT 10
            """
        elif relation_type == 'encodes':
            query = f"""
            PREFIX bio: <{PREFIX}>
            SELECT ?gene ?protein WHERE {{
                ?gene bio:encodes ?protein .
                FILTER(CONTAINS(STR(?gene), '{entity}') || CONTAINS(STR(?protein), '{entity}'))
            }} LIMIT 10
            """
        else:  # all
            query = f"""
            PREFIX bio: <{PREFIX}>
            PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
            SELECT ?subject ?predicate ?object WHERE {{
                ?subject ?predicate ?object .
                FILTER(CONTAINS(LCASE(STR(?subject)), LCASE('{entity}')) ||
                       CONTAINS(LCASE(STR(?object)), LCASE('{entity}')))
                FILTER(!isLiteral(?object) || isLiteral(?object))
            }} LIMIT 15
            """
        
        sparql.setQuery(query)
        results = sparql.query().convert()
        bindings = results['results']['bindings']
        
        if not bindings:
            return f'No se encontraron relaciones para "{entity}" en el Knowledge Graph.'
        
        lines = [f'📊 Knowledge Graph — Relaciones de "{entity}":']
        for b in bindings:
            s = b.get('subject', {}).get('value', '?').split('#')[-1]
            p = b.get('predicate', {}).get('value', '?').split('#')[-1]
            o = b.get('object', {}).get('value', '?').split('#')[-1]
            lines.append(f'  {s} --[{p}]--> {o}')
        
        return '\n'.join(lines)
    
    except Exception as e:
        return f'Error consultando GraphDB: {str(e)}. Verifique que GraphDB esté ejecutándose en localhost:7200'


# ────────────────────────────────────────────────────────────────────────────
# TOOL 3: HyDE — Transformación de consultas vagas
# ────────────────────────────────────────────────────────────────────────────
@tool
def apply_hyde_tool(short_query: str) -> str:
    """
    Aplica HyDE (Hypothetical Document Embeddings) a una consulta corta o ambigua.
    Genera un documento hipotético detallado y busca documentos similares a él.
    Usar cuando la consulta tiene menos de 10 palabras o es demasiado vaga.
    """
    hyde_prompt = ChatPromptTemplate.from_template("""
    Escribe un párrafo técnico biomédico detallado (150-200 palabras) que sería 
    la respuesta perfecta a esta consulta: {query}
    Párrafo:
    """)
    
    hyde_chain = hyde_prompt | gemini | StrOutputParser()
    hypothetical_doc = hyde_chain.invoke({'query': short_query})
    
    # Buscar con el documento hipotético
    retriever = faiss_index.as_retriever(
        search_type='mmr',
        search_kwargs={'k': 5, 'fetch_k': 20, 'lambda_mult': 0.6}
    )
    docs = retriever.invoke(hypothetical_doc)
    
    results = [f'[HyDE aplicado a: "{short_query}"]\n']
    for i, doc in enumerate(docs):
        results.append(f'[Doc {i+1}] {doc.metadata["doc_id"]}:\n{doc.page_content[:400]}')
    
    return '\n\n'.join(results)


# ────────────────────────────────────────────────────────────────────────────
# TOOL 4: Query Decomposition
# ────────────────────────────────────────────────────────────────────────────
@tool
def decompose_and_search(complex_query: str) -> str:
    """
    Descompone una consulta compleja en subconsultas y busca cada una por separado.
    Usar cuando la consulta contiene múltiples preguntas o condiciones (AND, OR, además).
    """
    decomp_prompt = ChatPromptTemplate.from_template("""
    Descompone la siguiente consulta compleja en 2-4 subconsultas simples separadas por '|'.
    Solo devuelve las subconsultas separadas por '|', nada más.
    
    Consulta compleja: {query}
    Subconsultas:
    """)
    
    decomp_chain = decomp_prompt | groq | StrOutputParser()
    sub_queries_str = decomp_chain.invoke({'query': complex_query})
    sub_queries = [sq.strip() for sq in sub_queries_str.split('|') if sq.strip()]
    
    all_results = [f'[Decomposición de: "{complex_query[:80]}..."]\n']
    seen_chunks = set()
    
    for i, sq in enumerate(sub_queries[:4]):
        all_results.append(f'--- Subconsulta {i+1}: "{sq}" ---')
        docs = faiss_index.similarity_search(sq, k=3)
        for doc in docs:
            chunk_id = doc.metadata['chunk_id']
            if chunk_id not in seen_chunks:
                seen_chunks.add(chunk_id)
                all_results.append(f'  [{doc.metadata["doc_id"]}]: {doc.page_content[:300]}')
    
    return '\n'.join(all_results)


# ────────────────────────────────────────────────────────────────────────────
# TOOL 5: Búsqueda en internet (fallback)
# ────────────────────────────────────────────────────────────────────────────
tavily_search = TavilySearchResults(
    max_results=5,
    search_depth='advanced',
    include_domains=['pubmed.ncbi.nlm.nih.gov', 'nature.com', 'sciencedirect.com'],
    description='Busca información biomédica actualizada en internet como fallback tras 3 intentos fallidos.'
)

# Lista de tools
TOOLS = [vector_search, knowledge_graph_query, apply_hyde_tool, decompose_and_search, tavily_search]

print('✅ 5 tools definidas:')
for t in TOOLS:
    print(f'   • {t.name}: {t.description[:70]}...')


## 2 · Estado del agente (LangGraph State)

In [ ]:
import operator
from langchain_core.messages import BaseMessage

class RAGAgentState(TypedDict):
    """Estado compartido entre todos los nodos del grafo LangGraph."""
    messages: Annotated[List[BaseMessage], operator.add]  # historial de mensajes
    original_query: str            # consulta original del usuario
    retrieved_docs: List[str]      # documentos recuperados
    current_answer: str            # respuesta actual generada
    reflection_score: float        # puntuación de la reflexión (0-1)
    reflection_feedback: str       # feedback del agente reflexivo
    attempt_count: int             # número de intentos de generación
    used_internet: bool            # si se usó búsqueda web
    trace: List[Dict[str, Any]]    # trazabilidad completa
    final_answer: str              # respuesta final al usuario

print('✅ Estado RAGAgentState definido')


## 3 · Nodos del grafo LangGraph

In [ ]:
# ── Nodo 1: Agente ReAct ──────────────────────────────────────────────────
# Razona y decide qué tools usar
REACT_SYSTEM = """Eres un experto en bioinformática y medicina que responde preguntas
usando un corpus de artículos científicos de arXiv.

Puedes usar las siguientes herramientas para encontrar información:
- vector_search: busca en el corpus biomédico por similitud semántica (MMR)
- knowledge_graph_query: consulta relaciones estructuradas en la ontología OWL
- apply_hyde_tool: para consultas cortas o vagas (< 10 palabras)
- decompose_and_search: para consultas complejas con múltiples preguntas
- tavily_search: SOLO como ÚLTIMO RECURSO después de 3 intentos fallidos

Antes de responder, SIEMPRE usa al menos una herramienta para recuperar contexto.
Cita siempre las fuentes de los documentos recuperados.
"""

react_llm = gemini.bind_tools(TOOLS)


def react_agent_node(state: RAGAgentState) -> RAGAgentState:
    """Nodo ReAct: razona y decide qué herramientas usar para responder la consulta."""
    messages = state['messages']
    
    # Añadir contexto del intento actual
    if state['attempt_count'] > 0:
        feedback_msg = SystemMessage(
            content=f'Intento {state["attempt_count"]+1}/3. Feedback anterior: {state["reflection_feedback"]}. '
                    f'Mejora tu respuesta usando diferentes herramientas o estrategias.'
        )
        messages = messages + [feedback_msg]
    
    # Invocar el agente ReAct (puede llamar a múltiples tools)
    response = react_llm.invoke(
        [SystemMessage(content=REACT_SYSTEM)] + messages
    )
    
    # Registrar en trazabilidad
    trace_entry = {
        'node': 'react_agent',
        'timestamp': datetime.now().isoformat(),
        'attempt': state['attempt_count'] + 1,
        'tool_calls': [tc['name'] for tc in (response.tool_calls or [])]
    }
    
    return {
        **state,
        'messages': [response],
        'trace': state.get('trace', []) + [trace_entry]
    }


# ── Nodo 2: Ejecutor de Tools ─────────────────────────────────────────────
tool_executor = ToolNode(TOOLS)


# ── Nodo 3: Generador de respuesta ────────────────────────────────────────
GENERATE_PROMPT = ChatPromptTemplate.from_messages([
    ('system', """
    Eres un asistente biomédico experto. Usando ÚNICAMENTE la información de los fragmentos
    recuperados, genera una respuesta completa, precisa y bien estructurada.
    
    REGLAS:
    - Cita las fuentes usando [doc_id, chunk_id]
    - Si la información es insuficiente, indícalo claramente
    - No inventes información no respaldada por los documentos
    - Estructura la respuesta con secciones si es una pregunta compleja
    """),
    ('human', 'Pregunta original: {query}\n\nContexto recuperado:\n{context}\n\nRespuesta:')
])

generate_chain = GENERATE_PROMPT | gemini | StrOutputParser()


def generate_response_node(state: RAGAgentState) -> RAGAgentState:
    """Nodo generador: construye la respuesta a partir del contexto recuperado."""
    # Extraer el contexto de los mensajes (resultados de tools)
    context_parts = []
    for msg in state['messages']:
        if hasattr(msg, 'content') and isinstance(msg.content, str):
            if len(msg.content) > 50:  # filtrar mensajes vacíos
                context_parts.append(msg.content)
    
    context = '\n\n---\n\n'.join(context_parts[-5:])  # últimos 5 fragmentos
    
    answer = generate_chain.invoke({
        'query': state['original_query'],
        'context': context
    })
    
    trace_entry = {
        'node': 'generate_response',
        'timestamp': datetime.now().isoformat(),
        'answer_length': len(answer),
        'context_fragments': len(context_parts)
    }
    
    return {
        **state,
        'current_answer': answer,
        'retrieved_docs': context_parts,
        'trace': state.get('trace', []) + [trace_entry]
    }


# ── Nodo 4: Reflexión / Crítica ───────────────────────────────────────────
REFLECT_PROMPT = ChatPromptTemplate.from_template("""
Eres un evaluador crítico de respuestas biomédicas. Evalúa la siguiente respuesta:

Pregunta original: {query}

Respuesta generada: {answer}

Contexto disponible: {context}

Evalúa en una escala 0-1 cada criterio:
1. RESPALDADA: ¿Está respaldada por el contexto? (0=sin respaldo, 1=completamente respaldada)
2. COHERENTE: ¿Es coherente y libre de contradicciones? 
3. COMPLETA: ¿Responde todos los aspectos de la pregunta?
4. CITACIONES: ¿Incluye citas a las fuentes?

Puntuación general (0-1): promedio de los 4 criterios
Feedback específico: qué debe mejorar

Responde en formato JSON:
{{
  "scores": {{"respaldada": 0.0, "coherente": 0.0, "completa": 0.0, "citaciones": 0.0}},
  "overall_score": 0.0,
  "feedback": "...",
  "approved": false
}}
""")


def reflect_node(state: RAGAgentState) -> RAGAgentState:
    """Nodo reflexivo: evalúa la calidad de la respuesta generada."""
    import json
    
    context = '\n'.join(state.get('retrieved_docs', [])[:3])[:2000]
    
    reflect_chain = REFLECT_PROMPT | gemini | StrOutputParser()
    raw_result = reflect_chain.invoke({
        'query': state['original_query'],
        'answer': state['current_answer'],
        'context': context
    })
    
    # Parsear JSON
    try:
        # Extraer JSON del texto
        import re
        json_match = re.search(r'\{.*\}', raw_result, re.DOTALL)
        if json_match:
            eval_result = json.loads(json_match.group())
        else:
            eval_result = {'overall_score': 0.5, 'feedback': raw_result, 'approved': False}
    except:
        eval_result = {'overall_score': 0.5, 'feedback': raw_result, 'approved': False}
    
    score = eval_result.get('overall_score', 0.5)
    feedback = eval_result.get('feedback', 'Sin feedback específico')
    approved = score >= 0.7  # umbral de aprobación
    
    trace_entry = {
        'node': 'reflect',
        'timestamp': datetime.now().isoformat(),
        'score': score,
        'approved': approved,
        'feedback': feedback[:200]
    }
    
    print(f'  🔍 Reflexión [intento {state["attempt_count"]+1}]: score={score:.2f} | approved={approved}')
    print(f'     Feedback: {feedback[:150]}...' if len(feedback) > 150 else f'     Feedback: {feedback}')
    
    return {
        **state,
        'reflection_score': score,
        'reflection_feedback': feedback,
        'attempt_count': state['attempt_count'] + 1,
        'trace': state.get('trace', []) + [trace_entry]
    }


# ── Nodo 5: Búsqueda web (fallback) ──────────────────────────────────────
def internet_fallback_node(state: RAGAgentState) -> RAGAgentState:
    """Nodo fallback: busca en internet tras 3 intentos fallidos y actualiza la KB."""
    print('  🌐 Activando búsqueda web (fallback tras 3 intentos fallidos)...')
    
    # Buscar en internet
    web_results = tavily_search.invoke(state['original_query'])
    
    # Formatear resultados web
    web_context = '\n\n'.join([
        f'[WEB] {r.get("url", "")}\n{r.get("content", "")}'
        for r in (web_results if isinstance(web_results, list) else [{'content': str(web_results)}])
    ])
    
    # Generar respuesta con contexto web
    web_answer = generate_chain.invoke({
        'query': state['original_query'],
        'context': f'[Fuente: Internet]\n{web_context}'
    })
    
    # Retroalimentar la base de conocimiento (agregar como nuevo documento)
    web_doc_id = f'web_fallback_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
    new_doc = {'page_content': web_context[:2000], 'metadata': {'doc_id': web_doc_id, 'source': 'internet'}}
    print(f'  📚 Conocimiento web indexado como: {web_doc_id}')
    
    trace_entry = {
        'node': 'internet_fallback',
        'timestamp': datetime.now().isoformat(),
        'web_sources': len(web_results) if isinstance(web_results, list) else 1
    }
    
    return {
        **state,
        'final_answer': web_answer,
        'current_answer': web_answer,
        'used_internet': True,
        'trace': state.get('trace', []) + [trace_entry]
    }


# ── Nodo 6: Finalización ──────────────────────────────────────────────────
def finalize_node(state: RAGAgentState) -> RAGAgentState:
    """Nodo final: prepara la respuesta final con trazabilidad completa."""
    return {
        **state,
        'final_answer': state['current_answer']
    }


print('✅ 6 nodos definidos')


## 4 · Construcción del grafo LangGraph

In [ ]:
# ── Funciones de enrutamiento ─────────────────────────────────────────────
def should_use_tools(state: RAGAgentState) -> str:
    """Determina si el agente debe llamar tools o pasar a generación."""
    last_msg = state['messages'][-1]
    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
        return 'use_tools'
    return 'generate'


def should_retry_or_finish(state: RAGAgentState) -> str:
    """Decide si reintentar, terminar con éxito o activar fallback web."""
    approved    = state['reflection_score'] >= 0.7
    max_retries = state['attempt_count'] >= 3
    
    if approved:
        return 'finalize'           # ✅ Respuesta aprobada
    elif max_retries:
        return 'internet_fallback'  # 🌐 Máx intentos: buscar en internet
    else:
        return 'retry'              # 🔄 Reintentar con feedback


# ── Construir el grafo ────────────────────────────────────────────────────
graph_builder = StateGraph(RAGAgentState)

# Añadir nodos
graph_builder.add_node('react_agent',       react_agent_node)
graph_builder.add_node('tools',             tool_executor)
graph_builder.add_node('generate_response', generate_response_node)
graph_builder.add_node('reflect',           reflect_node)
graph_builder.add_node('internet_fallback', internet_fallback_node)
graph_builder.add_node('finalize',          finalize_node)

# ── Edges (transiciones) ──────────────────────────────────────────────────
graph_builder.set_entry_point('react_agent')

# react_agent → [tools | generate] (según si hay tool_calls)
graph_builder.add_conditional_edges(
    'react_agent',
    should_use_tools,
    {'use_tools': 'tools', 'generate': 'generate_response'}
)

# tools → react_agent (el agente puede usar múltiples tools)
graph_builder.add_edge('tools', 'react_agent')

# generate_response → reflect
graph_builder.add_edge('generate_response', 'reflect')

# reflect → [finalize | retry | internet_fallback]
graph_builder.add_conditional_edges(
    'reflect',
    should_retry_or_finish,
    {
        'finalize':          'finalize',
        'retry':             'react_agent',
        'internet_fallback': 'internet_fallback'
    }
)

# internet_fallback → finalize
graph_builder.add_edge('internet_fallback', 'finalize')

# finalize → END
graph_builder.add_edge('finalize', END)

# Compilar
rag_graph = graph_builder.compile()

print('✅ Grafo LangGraph compilado')
print('   Flujo: react_agent → tools* → generate → reflect → [finalize | retry | web_fallback]')


## 5 · Función principal de ejecución

In [ ]:
def run_rag_agent(query: str, verbose: bool = True) -> dict:
    """
    Ejecuta el agente RAG agéntico completo (ReAct + Reflecting).
    
    Args:
        query: pregunta del usuario en lenguaje natural
        verbose: mostrar detalles del proceso
    
    Returns:
        dict con respuesta final, trazabilidad y métricas
    """
    if verbose:
        print(f'\n{'='*65}')
        print(f'🤖 Consulta: "{query}"')
        print('='*65)
    
    # Estado inicial
    initial_state: RAGAgentState = {
        'messages': [HumanMessage(content=query)],
        'original_query': query,
        'retrieved_docs': [],
        'current_answer': '',
        'reflection_score': 0.0,
        'reflection_feedback': '',
        'attempt_count': 0,
        'used_internet': False,
        'trace': [],
        'final_answer': ''
    }
    
    # Ejecutar el grafo
    final_state = rag_graph.invoke(initial_state)
    
    if verbose:
        print(f'\n📋 RESPUESTA FINAL:')
        print('-' * 65)
        print(final_state['final_answer'])
        print('-' * 65)
        print(f'\n📊 Estadísticas:')
        print(f'   Intentos: {final_state["attempt_count"]}')
        print(f'   Puntuación final: {final_state["reflection_score"]:.2f}')
        print(f'   Usó internet: {final_state["used_internet"]}')
        print(f'   Nodos ejecutados: {[t["node"] for t in final_state["trace"]]}')
    
    return {
        'query': query,
        'answer': final_state['final_answer'],
        'attempts': final_state['attempt_count'],
        'score': final_state['reflection_score'],
        'used_internet': final_state['used_internet'],
        'trace': final_state['trace']
    }


# ── Caso de uso 1: Consulta directa ──────────────────────────────────────
result1 = run_rag_agent(
    'What is the role of BRCA1 protein in DNA repair and why is it important in breast cancer?'
)


In [ ]:
# ── Caso de uso 2: Consulta compleja (espera Decomposition) ───────────────
result2 = run_rag_agent(
    'What genes are mutated in non-small cell lung cancer, what targeted therapies exist, and how does the EGFR pathway work?'
)


In [ ]:
# ── Caso de uso 3: Consulta corta (espera HyDE) ───────────────────────────
result3 = run_rag_agent('COVID-19 antiviral mechanisms')


## Resumen del grafo

```
Usuario
  ↓
react_agent (Gemini 2.0 Flash + 5 Tools)
  ↓ (si hay tool_calls)
tools [vector_search | kg_query | hyde | decompose | tavily]
  ↓
react_agent (ciclo hasta que no haya más tool_calls)
  ↓
generate_response (Gemini 2.0 Flash)
  ↓
reflect (evalúa score 0-1, umbral=0.7)
  ↓
┌─ score ≥ 0.7 → finalize → END
├─ score < 0.7 & intentos < 3 → retry → react_agent
└─ intentos = 3 → internet_fallback → finalize → END
```

**LLMs usados:**
- Gemini 2.0 Flash: razonamiento, generación, reflexión
- Groq LLaMA 3.3 70B: k dinámico, descomposición rápida